# Student Performance Dataset: Exploratory Data Analysis

This notebook inspects the real CSV supplied by the project. It creates no sample records and discovers all columns from the file. Run from the project root or the `notebooks` directory. Review target candidates before running target-dependent cells.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="crest")
possible_paths = [Path("data/student-performance.csv"), Path("../data/student-performance.csv")]
DATA_PATH = next((path for path in possible_paths if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Please place the Kaggle dataset at data/student-performance.csv")
df = pd.read_csv(DATA_PATH)
if df.empty:
    raise ValueError("The CSV contains no student records.")
df.columns = [str(column).strip() for column in df.columns]
print(f"Loaded {DATA_PATH}: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Dataset Overview

Review the actual schema, missingness, duplicates, and descriptive summaries before choosing an outcome.

In [ ]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)
display(overview)
print(f"Duplicate rows: {df.duplicated().sum():,}")
display(df.describe(include="all").T)

## Target Review

Only recognized numeric-compatible target names are suggested. If there is not exactly one candidate, set `target_column` explicitly after reviewing the printed columns.

In [ ]:
import re

target_aliases = {"final_score", "performance", "score", "g3", "final_grade", "grade"}
def normalize_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).casefold()).strip("_")
numeric_compatible = []
for column in df.columns:
    converted = pd.to_numeric(df[column], errors="coerce")
    if converted.notna().any() and converted.notna().sum() >= df[column].notna().sum() * 0.9:
        numeric_compatible.append(column)
target_candidates = [column for column in df.columns
                    if column in numeric_compatible and normalize_name(column) in target_aliases]
target_column = target_candidates[0] if len(target_candidates) == 1 else None
print("Numeric-compatible columns:", numeric_compatible)
print("Recognized target candidates:", target_candidates)
if target_column is None:
    print("No unique target selected. Set target_column to the intended numeric column before target analysis.")
else:
    print("Unique candidate selected:", target_column)

# Example after inspecting the candidates (edit only if needed):
# target_column = "your_actual_target_column"

## Feature Distributions

Plot numeric features as distributions and categorical features as bounded frequency charts to keep high-cardinality columns readable.

In [ ]:
numeric_features = []
for column in df.columns:
    values = pd.to_numeric(df[column], errors="coerce")
    observed = int(df[column].notna().sum())
    if pd.api.types.is_numeric_dtype(df[column]) or (
        observed > 0 and values.notna().sum() >= observed * 0.9
    ):
        numeric_features.append(column)
if target_column in numeric_features:
    numeric_features.remove(target_column)
categorical_features = [column for column in df.columns
                        if column not in numeric_features and column != target_column]
numeric_feature_data = pd.DataFrame(
    {column: pd.to_numeric(df[column], errors="coerce") for column in numeric_features},
    index=df.index,
)
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

if numeric_features:
    plot_columns = numeric_features[:12]
    numeric_feature_data[plot_columns].hist(
        figsize=(14, max(3, 2.8 * int(np.ceil(len(plot_columns) / 3)))),
        bins=25, layout=(int(np.ceil(len(plot_columns) / 3)), 3),
        color="#4b8274", edgecolor="white",
    )
    plt.suptitle("Numerical feature distributions", y=1.02)
    plt.tight_layout()
    plt.show()

for column in categorical_features[:6]:
    counts = df[column].astype("string").fillna("Missing").value_counts().head(15)
    if not counts.empty:
        plt.figure(figsize=(8, 3))
        sns.barplot(x=counts.values, y=counts.index, color="#d8784b")
        plt.title(f"{column}: most frequent values")
        plt.xlabel("Records")
        plt.ylabel("")
        plt.tight_layout()
        plt.show()

## Target Distribution and Relationships

These charts run only after a numeric target is selected. The correlation view uses numeric features; categorical comparisons are limited to columns with manageable cardinality.

In [ ]:
if target_column is None or target_column not in df.columns:
    print("Set target_column after reviewing the candidates to enable target analysis.")
else:
    target_values = pd.to_numeric(df[target_column], errors="coerce").dropna()
    if target_values.empty:
        raise ValueError("The selected target has no numeric values.")
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.histplot(target_values, bins=25, kde=True, ax=ax, color="#d8784b")
    ax.set_title(f"Distribution of {target_column}")
    plt.tight_layout()
    plt.show()
    display(target_values.describe().to_frame("target_summary"))

    numeric_for_corr = numeric_feature_data.copy()
    numeric_for_corr[target_column] = pd.to_numeric(df[target_column], errors="coerce")
    corr_with_target = numeric_for_corr.corr(numeric_only=True)[target_column].drop(target_column).dropna()
    if not corr_with_target.empty:
        display(corr_with_target.reindex(corr_with_target.abs().sort_values(ascending=False).index).to_frame("correlation"))
        top_correlated = corr_with_target.abs().nlargest(min(8, len(corr_with_target))).index.tolist()
        selected_x = top_correlated[0]
        fig = px.scatter(numeric_for_corr, x=selected_x, y=target_column, opacity=0.65,
                         title=f"{selected_x} vs {target_column}")
        fig.show()

    for column in categorical_features:
        if df[column].nunique(dropna=True) <= 15:
            grouped = df[[column, target_column]].copy()
            grouped[target_column] = pd.to_numeric(grouped[target_column], errors="coerce")
            grouped = grouped.dropna(subset=[target_column])
            if not grouped.empty:
                plt.figure(figsize=(max(7, df[column].nunique() * 0.7), 4))
                sns.boxplot(data=grouped, x=column, y=target_column, color="#85aaa0")
                plt.xticks(rotation=35, ha="right")
                plt.title(f"{target_column} by {column}")
                plt.tight_layout()
                plt.show()
                break

## Correlation and Outliers

Correlations are descriptive, not causal. Outlier counts use the conventional 1.5 × IQR rule and should be reviewed in context rather than automatically removed.

In [ ]:
numeric_for_corr = numeric_feature_data.copy()
if target_column in df.columns:
    numeric_for_corr[target_column] = pd.to_numeric(df[target_column], errors="coerce")
if numeric_for_corr.shape[1] >= 2:
    correlation = numeric_for_corr.corr(numeric_only=True)
    plt.figure(figsize=(max(8, correlation.shape[1] * 0.65), max(6, correlation.shape[1] * 0.55)))
    sns.heatmap(correlation, cmap="vlag", center=0, vmin=-1, vmax=1, linewidths=0.3)
    plt.title("Numerical feature correlation")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numerical columns are needed for a correlation heatmap.")

outlier_rows = []
for column in numeric_for_corr.columns:
    values = pd.to_numeric(numeric_for_corr[column], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    count = int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())
    outlier_rows.append({"feature": column, "iqr_outlier_count": count,
                        "iqr_outlier_pct": round(count / len(values) * 100, 2)})
outlier_summary = pd.DataFrame(outlier_rows).sort_values("iqr_outlier_count", ascending=False)
display(outlier_summary)